In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from math import sqrt

from sklearn.metrics import (mean_squared_error as MSE, mean_absolute_error as MAE, r2_score as R2,
                             explained_variance_score as EVS)
from sklearn.preprocessing import RobustScaler

from keras.models import load_model

%load_ext autoreload
%autoreload 2

%matplotlib inline

plt.rcParams['figure.figsize'] = (10, 8)

/home/bulent/anaconda3/lib/python3.6/site-packages/h5py/__init__.py:34: FutureWarning: Conversion of the second argument of issubdtype from `float` to `np.floating` is deprecated. In future, it will be treated as `np.float64 == np.dtype(float).type`.
  from ._conv import register_converters as _register_converters
Using TensorFlow backend.


In [2]:
def fraction_within_eps(y_true, y_pred, epsilon=0.5):
    # fraction of entries where abs(y_true - y_pred) < epsilon
    
    count = np.sum(np.abs(y_true - y_pred) <= epsilon)
    return count / y_true.shape[0]

yt = np.array([10, 10, 10, 10, 10])
yp = np.array([9.2, 9.7, 10.3, 10.2, 11])
print(fraction_within_eps(yt, yp, 0.6))
FIE = fraction_within_eps

0.6


In [3]:
def concorr(x, y):
    # Return Lin's concordance correlation coefficient
    # x, y are numpy arrays
    
    xm = x.mean()
    ym = y.mean()
    xv = x.var()
    yv = y.var()
    xycov = np.sum((x-xm)*(y-ym)) / x.shape[0]
    lin = 2*xycov / (xv + yv + (xm - ym)**2)
    return lin

xx = np.random.randn(10)
yy = xx + 10
print('R2 corr coef is {} whereas concordance corr coef is {}'.format(R2(yy,xx), concorr(yy,xx)))

R2 corr coef is -145.0961193518828 whereas concordance corr coef is 0.013504742789700737


In [2]:
with open('stations-6to31.pkl', 'rb') as f:
    datas = pickle.load(f)
    
x_train6_ = datas['x_train6']
y_train6 = datas['y_train6']
x_train_ = datas['x_train']
y_train = datas['y_train']
x_dev_ = datas['x_dev']
y_dev = datas['y_dev']
x_test_ = datas['x_test']
y_test = datas['y_test']

print(f'x_train shape: {x_train_.shape}, y_train shape: {y_train.shape}')
print(f'x_train6 shape: {x_train6_.shape}, y_train6 shape: {y_train6.shape}')
print(f'x_dev shape: {x_dev_.shape}, y_dev shape: {y_dev.shape}')
print(f'x_test shape: {x_test_.shape}, y_test shape: {y_test.shape}')

x_train shape: (52416, 5), y_train shape: (52416,)
x_train6 shape: (10654, 5), y_train6 shape: (10654,)
x_dev shape: (9011, 5), y_dev shape: (9011,)
x_test shape: (16899, 5), y_test shape: (16899,)


In [5]:
def scale_data(scaler, datas):
    # scaler is a scaling function from sklearn library
    # datas is a dictionary, containing 3 sets of x_data with keys - x_train, x_dev, x_test
    # fit on x_train and return the transformed sets of data
    
    x_train = scaler.fit_transform(datas['x_train'].astype(float))
    x_dev = scaler.transform(datas['x_dev'].astype(float))
    x_test = scaler.transform(datas['x_test'].astype(float))
    
    transformed = {'x_train': x_train, 'x_dev': x_dev, 'x_test': x_test}
    return transformed

data6_ = {'x_train': x_train6_, 'x_dev': x_dev_, 'x_test': x_test_}
data_ = {'x_train': x_train_, 'x_dev': x_dev_, 'x_test': x_test_}

data6 = scale_data(RobustScaler(), data6_)
data = scale_data(RobustScaler(), data_)

x_train6 = data6['x_train']
x_train = data['x_train']

x_dev6 = data6['x_dev']
x_dev = data['x_dev']

x_test6 = data6['x_test']
x_test = data['x_test']

print(f'x_train shape: {x_train.shape}, y_train shape: {y_train.shape}')
print(f'x_train6 shape: {x_train6.shape}, y_train6 shape: {y_train6.shape}')
print(f'x_dev shape: {x_dev.shape}, y_dev shape: {y_dev.shape}')
print(f'x_test shape: {x_test.shape}, y_test shape: {y_test.shape}')

x_train shape: (52416, 5), y_train shape: (52416,)
x_train6 shape: (10654, 5), y_train6 shape: (10654,)
x_dev shape: (9011, 5), y_dev shape: (9011,)
x_test shape: (16899, 5), y_test shape: (16899,)


In [6]:
data6_2v_ = {'x_train': x_train6_[:, [1, 3]], 'x_dev': x_dev_[:, [1, 3]], 'x_test': x_test_[:, [1, 3]]}
data_2v_ = {'x_train': x_train_[:, [1, 3]], 'x_dev': x_dev_[:, [1, 3]], 'x_test': x_test_[:, [1, 3]]}

data6_2v = scale_data(RobustScaler(), data6_2v_)
data_2v = scale_data(RobustScaler(), data_2v_)

x_train6_2v = data6_2v['x_train']
x_train_2v = data_2v['x_train']

x_dev6_2v = data6_2v['x_dev']
x_dev_2v = data_2v['x_dev']

x_test6_2v = data6_2v['x_test']
x_test_2v = data_2v['x_test']

y_train6_hh0 = y_train6 / x_train6_[:, 4]
y_train_hh0 = y_train / x_train_[:, 4]
y_dev_hh0 = y_dev / x_dev_[:, 4]
y_test_hh0 = y_test / x_test_[:, 4]

print(f'x_train shape: {x_train_2v.shape}, y_train shape: {y_train_hh0.shape}')
print(f'x_train6 shape: {x_train6_2v.shape}, y_train6 shape: {y_train6_hh0.shape}')
print(f'x_dev shape: {x_dev_2v.shape}, y_dev shape: {y_dev_hh0.shape}')
print(f'x_test shape: {x_test_2v.shape}, y_test shape: {y_test_hh0.shape}')

x_train shape: (52416, 2), y_train shape: (52416,)
x_train6 shape: (10654, 2), y_train6 shape: (10654,)
x_dev shape: (9011, 2), y_dev shape: (9011,)
x_test shape: (16899, 2), y_test shape: (16899,)


In [13]:
print(y_test[0])
print(x_test_[1200,4])

7.135182
34.29094534400971


In [14]:
def quadratic(s):
    return 0.145 + 0.845*s - 0.280*s**2

#print(x_test_[5:10]) # - phi - n - T - N - H0 are the column order
h0 = x_test_[:, 4]
s = x_test_[:, 1] / x_test_[:, 3]

p = h0 * quadratic(s)

mse = MSE( y_test, p )
rmse = sqrt( mse )
mae = MAE( y_test, p )
r2 = R2( y_test, p )
evs = EVS( y_test, p )
fie_h = FIE( y_test, p, 0.5 )
fie_o = FIE( y_test, p, 1 )
fie_oh = FIE( y_test, p, 1.5 )
concor = concorr(y_test, p )
print('« Quadratic in s with Test Set »\nRMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '
              'FIE_H: {:.4f}, FIE_O: {:.4f}, FIE_OH: {:.4f}'.format(rmse, mae, r2, evs, fie_h, fie_o, fie_oh))
print(f'Lin\'s Concordance Correlation: {concor} ')

« Quadratic in s with Test Set »
RMSE: 6.2895, MAE: 2.5645, R2: 0.4616, EVS: 0.5120, FIE_H: 0.1737, FIE_O: 0.3439, FIE_OH: 0.5000
Lin's Concordance Correlation: 0.7624915439373603 


In [36]:
from scipy.optimize import curve_fit

def angstrom(xdata, a, b, c):
    # xdata is just s = n / N
    s = xdata
    # a*T + b*T^2 + c*s + d*s^2 + e
    hh0 = a*s + b*s**2 + c
    return hh0

h0 = x_test_[:, 4]
s = x_test_[:, 1] / x_test_[:, 3]

h0_train6 = x_train6_[:, 4]
s_train6 = x_train6_[:, 1] / x_train6_[:, 3]

h0_train = x_train_[:, 4]
s_train = x_train_[:, 1] / x_train_[:, 3]

popt6, _ = curve_fit(angstrom, s_train6, y_train6/h0_train6)
popt, _ = curve_fit(angstrom, s_train, y_train/h0_train)

p6 = h0 * angstrom(s, *popt6)
p = h0 * angstrom(s, *popt)


mse = MSE( y_test, p6 )
rmse = sqrt( mse )
mae = MAE( y_test, p6 )
r2 = R2( y_test, p6 )
evs = EVS( y_test, p6 )
fie_h = FIE( y_test, p6, 0.5 )
fie_o = FIE( y_test, p6, 1 )
fie_oh = FIE( y_test, p6, 1.5 )
concor = concorr( y_test, p6 )
print('« Angstrom Model trained with 6 stations, Test Set Perf »\nRMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '
              'FIE_H: {:.4f}, FIE_O: {:.4f}, FIE_OH: {:.4f}'.format(rmse, mae, r2, evs, fie_h, fie_o, fie_oh))
print(f'Lin\'s Concordance Correlation: {concor} ')



mse = MSE( y_test, p )
rmse = sqrt( mse )
mae = MAE( y_test, p )
r2 = R2( y_test, p )
evs = EVS( y_test, p )
fie_h = FIE( y_test, p, 0.5 )
fie_o = FIE( y_test, p, 1 )
fie_oh = FIE( y_test, p, 1.5 )
concor = concorr( y_test, p )
print('« Angstrom Model with Test Set »\nRMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '
              'FIE_H: {:.4f}, FIE_O: {:.4f}, FIE_OH: {:.4f}'.format(rmse, mae, r2, evs, fie_h, fie_o, fie_oh))
print(f'Lin\'s Concordance Correlation: {concor} ')

« Angstrom Model trained with 6 stations, Test Set Perf »
RMSE: 4.0293, MAE: 2.0793, R2: 0.7790, EVS: 0.7943, FIE_H: 0.2437, FIE_O: 0.4523, FIE_OH: 0.6081
Lin's Concordance Correlation: 0.8849593555399916 
« Angstrom Model with Test Set »
RMSE: 3.8588, MAE: 2.0309, R2: 0.7973, EVS: 0.8092, FIE_H: 0.2477, FIE_O: 0.4622, FIE_OH: 0.6171
Lin's Concordance Correlation: 0.8933090982680278 


In [18]:
from scipy.optimize import curve_fit

def wololo(xdata, a, b, c, d, e):
    # xdata must be a tuple
    s, tavg = xdata
    # a*T + b*T^2 + c*s + d*s^2 + e
    hh0 = a * tavg + b * tavg ** 2 + c * s + d * s ** 2 + e
    return hh0

h0_train6 = x_train6_[:, 4]
s_train6 = x_train6_[:, 1] / x_train6_[:, 3]
tavg_train6 = x_train6_[:, 2]
xdat_train6 = (s_train6, tavg_train6)

popt6, _ = curve_fit(wololo, xdat_train6, y_train6/h0_train6)


h0_train = x_train_[:, 4]
s_train = x_train_[:, 1] / x_train_[:, 3]
tavg_train = x_train_[:, 2]
xdat_train = (s_train, tavg_train)

popt, _ = curve_fit(wololo, xdat_train, y_train/h0_train)

tavg = x_test_[:, 2]
xdat = (s, tavg)

p6 = h0 * wololo(xdat, *popt6)

mse = MSE( y_test, p6 )
rmse = sqrt( mse )
mae = MAE( y_test, p6 )
r2 = R2( y_test, p6 )
evs = EVS( y_test, p6 )
fie_h = FIE( y_test, p6, 0.5 )
fie_o = FIE( y_test, p6, 1 )
fie_oh = FIE( y_test, p6, 1.5 )
concor = concorr( y_test, p6 )
print('« Quadratic in s,Tavg trained with 6 stations Test Set Perf »\nRMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '
              'FIE_H: {:.4f}, FIE_O: {:.4f}, FIE_OH: {:.4f}'.format(rmse, mae, r2, evs, fie_h, fie_o, fie_oh))
print(f'Lin\'s Concordance Correlation: {concor} ')



p = h0 * wololo(xdat, *popt)

mse = MSE( y_test, p )
rmse = sqrt( mse )
mae = MAE( y_test, p )
r2 = R2( y_test, p )
evs = EVS( y_test, p )
fie_h = FIE( y_test, p, 0.5 )
fie_o = FIE( y_test, p, 1 )
fie_oh = FIE( y_test, p, 1.5 )
concor = concorr( y_test, p )
print('« Quadratic in s,Tavg trained with 31 stations Test Set Perf »\nRMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '
              'FIE_H: {:.4f}, FIE_O: {:.4f}, FIE_OH: {:.4f}'.format(rmse, mae, r2, evs, fie_h, fie_o, fie_oh))
print(f'Lin\'s Concordance Correlation: {concor} ')

« Quadratic in s,Tavg trained with 6 stations Test Set Perf »
RMSE: 4.0728, MAE: 2.1512, R2: 0.7742, EVS: 0.7928, FIE_H: 0.2347, FIE_O: 0.4277, FIE_OH: 0.5815
Lin's Concordance Correlation: 0.8779379261072009 
« Quadratic in s,Tavg trained with 31 stations Test Set Perf »
RMSE: 3.9457, MAE: 2.0692, R2: 0.7881, EVS: 0.8026, FIE_H: 0.2450, FIE_O: 0.4475, FIE_OH: 0.6052
Lin's Concordance Correlation: 0.8861007701905707 


In [42]:
err = (y_test * 0.02) ** 2
rmse = np.sqrt(np.sum(err) / len(y_test))
print(rmse)

0.3754332596308693


In [43]:
with open('measures_6to31.pkl', 'rb') as f:
    measures = pickle.load(f)
    
mess = ['rmse', 'mae', 'r2', 'evs', 'fie_h', 'fie_o', 'fie_oh', 'lin']
test6 = np.array(measures['te6']).T # 8x50
test31 = np.array(measures['te31']).T
for i in range(8):
    exec(f'{mess[i]} = np.array([test6[i], test31[i]])')
    exec(f'{mess[i]} = {mess[i]}.T')
    
print(rmse.shape)

(50, 2)


In [51]:
# mean perf wrt rmse, mae, r2 and lincc of NN trained with 31 stations
mean_perfs = test31.mean(axis=1)
print(mean_perfs)
max_perfs = test31.max(axis=1)
print(max_perfs)
min_perfs = test31.min(axis=1)
print(min_perfs)

[3.87469807 1.94258443 0.79561562 0.79915408 0.27733949 0.5044618
 0.65922599 0.89603209]
[3.99386708 1.9754534  0.80584311 0.80750975 0.2861116  0.51547429
 0.66619327 0.89987533]
[3.77679422 1.912998   0.78288322 0.78673865 0.27031185 0.4945263
 0.65015681 0.88915438]


In [52]:
# mean perf wrt rmse, mae, r2 and lincc of NN trained with 31 stations
mean_perfs = test6.mean(axis=1)
print(mean_perfs)
max_perfs = test6.max(axis=1)
print(max_perfs)
min_perfs = test6.min(axis=1)
print(min_perfs)

[3.94409443 1.97414406 0.78815568 0.79320929 0.27104326 0.49424936
 0.65125865 0.89042781]
[4.16575332 2.02842881 0.80520183 0.80999777 0.27841884 0.50535535
 0.66104503 0.8979294 ]
[3.78302626 1.93943999 0.76379273 0.76792697 0.25486715 0.47269069
 0.62790698 0.87915842]
